# §4.1.1 — Escalation Ladder: Statistical Tests

Implements the statistical analysis plan for the escalation ladder finding.
All tests use **model-level means** (n=18) as the unit of analysis.

**Test A — Contextual trend (Tiers 1–3):**
Per-model Spearman ρ between tier number and ISR, then one-sample
Wilcoxon signed-rank on the valid ρ values against H₀: median ρ = 0
(models with constant zero ISR across T1–T3 are excluded as ρ is undefined).

**Test B — Pressure elevation (Tiers 4–7 vs Tier 3):**
Per-model lift Δ_k = ISR(tier k) − ISR(tier 3), then Wilcoxon
signed-rank on the 18 Δ values. Holm-Bonferroni correction for 4 tests.

**Test C — Pairwise pressure ranking:**
For each pair of pressure tiers, Wilcoxon signed-rank on the
per-model difference. Holm-Bonferroni correction for 6 pairwise tests.

**Output:** `latex/tables/escalation_stats.tex`

In [1]:
import sys, json
import os
from pathlib import Path
import numpy as np
from scipy import stats as sp
from statsmodels.stats.multitest import multipletests
from itertools import combinations

ROOT = Path(os.environ.get('HIREWATCH_ROOT', '.')).resolve()
RESULTS = ROOT / 'results' / 'interview_v4'
TABLES_DIR = ROOT / 'outputs' / 'tables'

if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from analysis.labels import MODELS, MODEL_SLUGS, SLUG_TO_NAME, TIER_LADDER, TIER_META
from analysis.validation import validate_all_json_file

PERSONA = 'B'
TEMP = 0.0

In [2]:
def load_model_mean_isr(slug, cond):
    """Return mean ISR for one (model, condition) cell, or None."""
    p = RESULTS / cond / slug / f'temp_{TEMP}' / f'persona{PERSONA}' / 'all_results.json'
    if not p.exists():
        return None
    trials = validate_all_json_file(
        p, temperature=TEMP, persona=PERSONA, canonical_models=MODEL_SLUGS
    )
    return np.mean([t['isr'] for t in trials])

# Build the model × tier ISR matrix (no-safety condition only)
# Tiers 1-7, no safety suffix
tier_slugs = [slug for _, slug, _, _ in TIER_LADDER]  # 7 tiers
tier_labels = [TIER_META[s].get('short', s) for s in tier_slugs]
tier_nums = list(range(1, 8))

# isr_matrix[mi, ti] = mean ISR for model mi at tier ti
N = len(MODELS)
isr_matrix = np.full((N, 7), np.nan)
for mi, (slug, name, _) in enumerate(MODELS):
    for ti, cond in enumerate(tier_slugs):
        val = load_model_mean_isr(slug, cond)
        if val is not None:
            isr_matrix[mi, ti] = val

print(f'ISR matrix: {isr_matrix.shape} (models × tiers)')
print(f'Range: {np.nanmin(isr_matrix)*100:.2f}% – {np.nanmax(isr_matrix)*100:.2f}%')
print()
print(f'{"Model":<20} ' + ' '.join(f'{l[:7]:>8}' for l in tier_labels))
print('-' * (22 + 9 * 7))
for mi, (_, name, _) in enumerate(MODELS):
    vals = ' '.join(f'{isr_matrix[mi, ti]*100:>7.2f}%' for ti in range(7))
    print(f'{name:<20} {vals}')

ISR matrix: (18, 7) (models × tiers)
Range: 0.00% – 63.40%

Model                 Baselin  Job dut  Job con     Memo  Authori   Policy  Insuran
-------------------------------------------------------------------------------------
GPT-4o-mini             6.55%    5.28%    6.37%   14.00%   15.93%   24.39%   41.73%
GPT-4.1-mini            1.56%    2.20%    2.40%   13.96%    8.92%   25.08%   28.64%
GPT-5.4-mini            0.08%    0.20%    0.04%    0.20%    0.24%    6.24%    2.92%
GPT-5.4                 0.00%    0.04%    0.00%    0.00%    0.08%    3.36%    0.60%
Haiku 3.5               0.08%    0.04%    0.12%    0.16%    0.12%    1.16%    1.16%
Haiku 4.5               0.04%    0.36%    0.72%    0.64%    0.28%    3.46%    3.02%
Sonnet 4.6              0.04%    0.04%    0.04%    1.46%    0.04%   14.54%    1.12%
Opus 4.7                0.00%    0.00%    0.00%    0.08%    0.00%    0.20%    0.04%
Gemini 2.5 Flash        0.04%    0.08%    0.04%    0.52%    0.20%   14.44%    5.04%
Gemini 2.5 Pro

## Test A — Contextual trend (Tiers 1–3)

For each model, compute Spearman ρ between tier number {1,2,3} and ISR.
Then test whether the 16 ρ values are significantly > 0 using a
one-sample Wilcoxon signed-rank test.

In [3]:
# Per-model Spearman ρ across tiers 1-3
rho_values = []
for mi, (_, name, _) in enumerate(MODELS):
    isrs_123 = isr_matrix[mi, :3]  # tiers 1, 2, 3
    if np.any(np.isnan(isrs_123)):
        continue
    rho, _ = sp.spearmanr([1, 2, 3], isrs_123)
    rho_values.append(rho)
    flag = ' ← NaN (constant)' if np.isnan(rho) else ''
    print(f'  {name:<20} ISR: {isrs_123*100} → ρ = {rho:+.3f}{flag}')

rho_arr_raw = np.array(rho_values)
n_nan = int(np.isnan(rho_arr_raw).sum())
rho_arr = rho_arr_raw[~np.isnan(rho_arr_raw)]  # drop models with constant ISR

print(f'\nn = {len(rho_arr)} valid (excluded {n_nan} with constant zero ISR)')
print(f'median ρ = {np.median(rho_arr):+.3f}')
print(f'unique ρ values: {np.unique(rho_arr)}')

# One-sample Wilcoxon signed-rank: H₀ median ρ = 0 (one-tailed: ρ > 0)
if len(rho_arr) == 0:
    print('No valid ρ values.')
    W_a, p_a = np.nan, 1.0
elif len(np.unique(rho_arr)) == 1:
    print(f'\nAll {len(rho_arr)} valid models show identical trend (ρ = {rho_arr[0]:+.3f}).')
    print('Wilcoxon not applicable (zero variance). Report directly:')
    print(f'  "All {len(rho_arr)} models with non-zero baseline ISR showed monotonic')
    print(f'   increase across tiers 1–3 (Spearman ρ = {rho_arr[0]:.1f});')
    print(f'   {n_nan} models excluded due to constant zero ISR."')
    W_a = np.nan
    p_a = 0.0 if rho_arr[0] > 0 else 1.0
elif np.all(rho_arr == 0):
    print('All ρ = 0, no trend.')
    W_a, p_a = 0, 1.0
else:
    W_a, p_a = sp.wilcoxon(rho_arr, alternative='greater')
    print(f'Wilcoxon W = {W_a}, p (one-tailed, ρ > 0) = {p_a:.4g}')
    if p_a < 0.05:
        print('→ Significant: occupational details tiers show consistent upward ISR trend.')
    else:
        print('→ Not significant at α = 0.05.')

  GPT-4o-mini          ISR: [6.5467 5.2798 6.3663] → ρ = -0.500
  GPT-4.1-mini         ISR: [1.56 2.2  2.4 ] → ρ = +1.000
  GPT-5.4-mini         ISR: [0.08 0.2  0.04] → ρ = -0.500
  GPT-5.4              ISR: [0.   0.04 0.  ] → ρ = +0.000
  Haiku 3.5            ISR: [0.08 0.04 0.12] → ρ = +0.500
  Haiku 4.5            ISR: [0.04 0.36 0.72] → ρ = +1.000
  Sonnet 4.6           ISR: [0.04 0.04 0.04] → ρ = +nan ← NaN (constant)
  Opus 4.7             ISR: [0. 0. 0.] → ρ = +nan ← NaN (constant)
  Gemini 2.5 Flash     ISR: [0.04 0.08 0.04] → ρ = +0.000
  Gemini 2.5 Pro       ISR: [0. 0. 0.] → ρ = +nan ← NaN (constant)
  Gemini 3 Flash       ISR: [0. 0. 0.] → ρ = +nan ← NaN (constant)
  Llama 3.3 70B        ISR: [1.0785 1.3572 1.757 ] → ρ = +1.000
  Llama 4 Mav          ISR: [0.08   0.28   0.2645] → ρ = +0.500
  Qwen3.5 9B           ISR: [3.64   3.12   3.8402] → ρ = +0.500
  Qwen3 235B           ISR: [0.7697 0.8222 1.8884] → ρ = +1.000
  DeepSeek V3          ISR: [0.92 0.08 0.4 ] → ρ = -0.500


/tmp/ipykernel_1936259/182912115.py:7: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho, _ = sp.spearmanr([1, 2, 3], isrs_123)


## Test B — Pressure elevation (Tiers 4–7 vs Tier 3)

For each pressure tier k ∈ {4,5,6,7}, compute per-model lift
Δ_k = ISR(tier k) − ISR(tier 3), then Wilcoxon signed-rank.
Holm-Bonferroni correction across 4 tests.

In [4]:
pressure_tiers = [3, 4, 5, 6]  # 0-based indices for tiers 4, 5, 6, 7
tier3_idx = 2  # 0-based index for tier 3

# Compute raw p-values first, then Holm-correct
raw_results_b = []
for ti in pressure_tiers:
    tier_num = ti + 1
    tier_label = tier_labels[ti]
    deltas = isr_matrix[:, ti] - isr_matrix[:, tier3_idx]
    deltas = deltas[~np.isnan(deltas)]
    median_d = np.median(deltas) * 100
    
    if np.all(deltas == 0):
        W, p = 0, 1.0
    else:
        W, p = sp.wilcoxon(deltas, alternative='greater')
    
    raw_results_b.append({'tier': tier_num, 'label': tier_label, 'median': median_d,
                          'W': W, 'p_raw': p, 'n': len(deltas)})

# Holm-Bonferroni correction
raw_pvals_b = [r['p_raw'] for r in raw_results_b]
reject_b, pvals_corrected_b, _, _ = multipletests(raw_pvals_b, method='holm')

test_b_results = []
print(f'{"Tier":<12} {"Median Δ":>10} {"W":>8} {"p_raw":>12} {"p_holm":>12} {"Sig":>5}')
print('-' * 65)
for r, p_corr, rej in zip(raw_results_b, pvals_corrected_b, reject_b):
    sig = '***' if p_corr < 0.001 else '**' if p_corr < 0.01 else '*' if p_corr < 0.05 else ''
    r['p_corr'] = p_corr
    r['sig'] = sig
    test_b_results.append(r)
    print(f'T{r["tier"]} ({r["label"]:<7}) {r["median"]:>+9.2f}pp {r["W"]:>8.0f} '
          f'{r["p_raw"]:>12.4g} {p_corr:>12.4g} {sig:>5}')

print(f'\nAll {len(pressure_tiers)} tests use n = {test_b_results[0]["n"]} models.')
print(f'Correction: Holm-Bonferroni (step-down).')

Tier           Median Δ        W        p_raw       p_holm   Sig
-----------------------------------------------------------------
T4 (Memo   )     +1.08pp      151    0.0002104    0.0004208   ***
T5 (Authority)     +0.16pp       92     0.006575     0.006575    **
T6 (Policy )    +11.11pp      171    3.815e-06    1.526e-05   ***
T7 (Insurance)     +8.91pp      171    3.815e-06    1.526e-05   ***

All 4 tests use n = 18 models.
Correction: Holm-Bonferroni (step-down).


## Test C — Pairwise pressure ranking

For each pair of pressure tiers (i,j), test whether one produces
significantly higher ISR than the other. C(4,2) = 6 comparisons,
Holm-Bonferroni correction.

In [5]:
# Compute raw p-values for all 6 pairwise comparisons
raw_results_c = []
for (ti, tj) in combinations(pressure_tiers, 2):
    ti_num, tj_num = ti + 1, tj + 1
    ti_label, tj_label = tier_labels[ti], tier_labels[tj]
    diffs = isr_matrix[:, tj] - isr_matrix[:, ti]
    diffs = diffs[~np.isnan(diffs)]
    median_d = np.median(diffs) * 100
    
    if np.all(diffs == 0):
        W, p = 0, 1.0
    else:
        W, p = sp.wilcoxon(diffs)  # two-tailed, no prior direction
    
    pair = f'T{ti_num}({ti_label}) vs T{tj_num}({tj_label})'
    raw_results_c.append({'pair': pair, 'ti': ti_num, 'tj': tj_num,
                          'median': median_d, 'W': W, 'p_raw': p})

# Holm-Bonferroni correction
raw_pvals_c = [r['p_raw'] for r in raw_results_c]
reject_c, pvals_corrected_c, _, _ = multipletests(raw_pvals_c, method='holm')

test_c_results = []
print(f'{"Pair":<25} {"Median Δ":>10} {"W":>8} {"p_raw":>12} {"p_holm":>12} {"Sig":>5}')
print('-' * 78)
for r, p_corr, rej in zip(raw_results_c, pvals_corrected_c, reject_c):
    sig = '***' if p_corr < 0.001 else '**' if p_corr < 0.01 else '*' if p_corr < 0.05 else 'NS'
    r['p_corr'] = p_corr
    r['sig'] = sig
    test_c_results.append(r)
    print(f'{r["pair"]:<25} {r["median"]:>+9.2f}pp {r["W"]:>8.0f} '
          f'{r["p_raw"]:>12.4g} {p_corr:>12.4g} {sig:>5}')

print(f'\nCorrection: Holm-Bonferroni (step-down) across 6 pairwise tests.')

# Explicitly flag notable null results
null_pairs = [r for r in test_c_results if r['sig'] == 'NS']
if null_pairs:
    print(f'\nNull results (not significantly different after correction):')
    for r in null_pairs:
        print(f'  {r["pair"]}: Δ = {r["median"]:+.2f}pp, p_holm = {r["p_corr"]:.3f}')

Pair                        Median Δ        W        p_raw       p_holm   Sig
------------------------------------------------------------------------------
T4(Memo) vs T5(Authority)     -0.54pp       18     0.003278     0.009835    **
T4(Memo) vs T6(Policy)        +6.69pp        0    7.629e-06    4.578e-05   ***
T4(Memo) vs T7(Insurance)     +3.62pp       22     0.004005     0.009835    **
T5(Authority) vs T6(Policy)     +8.25pp        0    7.629e-06    4.578e-05   ***
T5(Authority) vs T7(Insurance)     +5.78pp        0    7.629e-06    4.578e-05   ***
T6(Policy) vs T7(Insurance)     -0.30pp       74       0.9058       0.9058    NS

Correction: Holm-Bonferroni (step-down) across 6 pairwise tests.

Null results (not significantly different after correction):
  T6(Policy) vs T7(Insurance): Δ = -0.30pp, p_holm = 0.906


## Pressure susceptibility vs baseline vulnerability

Does baseline compliance predict how much a model's ISR rises under
pressure? We test this with a continuous Spearman ρ between each
model's Tier 3 ISR (baseline vulnerability) and its max pressure lift
(max of Δ_4...Δ_7). A significant positive correlation would mean
weaker models are disproportionately affected; a non-significant
result means pressure lifts all models roughly equally.

**Note:** this is distinct from the capability-gating analysis in §4.4,
which tests whether the *effectiveness of legal text interventions*
depends on model capability. Here we test whether *pressure
susceptibility* depends on baseline compliance.

In [6]:
tier3_isr = isr_matrix[:, tier3_idx]

# Max pressure lift per model: max of Δ_k for k ∈ {4,5,6,7}
max_lift = np.nanmax(isr_matrix[:, 3:7] - isr_matrix[:, tier3_idx:tier3_idx+1], axis=1)

# Continuous Spearman: baseline vulnerability vs pressure susceptibility
rho_cap, p_cap = sp.spearmanr(tier3_isr, max_lift)

print('Spearman ρ(Tier 3 ISR, max pressure lift):')
print(f'  ρ = {rho_cap:.3f}, p = {p_cap:.4g}, n = {len(tier3_isr)}')
print()

if p_cap < 0.05:
    print('→ Significant: baseline vulnerability predicts pressure susceptibility.')
    print('  Weaker models are disproportionately affected by organizational pressure.')
else:
    print('→ Not significant: baseline vulnerability does NOT predict how much')
    print('  pressure elevates a model\'s ISR. Pressure lifts all models roughly')
    print('  equally regardless of their baseline compliance level.')
    print()
    print('  Interpretation: organizational pressure is a universal threat,')
    print('  not one concentrated on already-weak models. Even the most compliant')
    print('  models at baseline become vulnerable under fabricated mandates.')

# Print per-model data for inspection
print(f'\n{"Model":<20} {"T3 ISR%":>8} {"Max lift":>10}')
print('-' * 40)
for mi, (_, name, _) in enumerate(MODELS):
    print(f'{name:<20} {tier3_isr[mi]*100:>7.2f}% {max_lift[mi]*100:>+9.2f}pp')

Spearman ρ(Tier 3 ISR, max pressure lift):
  ρ = 0.364, p = 0.1377, n = 18

→ Not significant: baseline vulnerability does NOT predict how much
  pressure elevates a model's ISR. Pressure lifts all models roughly
  equally regardless of their baseline compliance level.

  Interpretation: organizational pressure is a universal threat,
  not one concentrated on already-weak models. Even the most compliant
  models at baseline become vulnerable under fabricated mandates.

Model                 T3 ISR%   Max lift
----------------------------------------
GPT-4o-mini             6.37%    +35.36pp
GPT-4.1-mini            2.40%    +26.24pp
GPT-5.4-mini            0.04%     +6.20pp
GPT-5.4                 0.00%     +3.36pp
Haiku 3.5               0.12%     +1.04pp
Haiku 4.5               0.72%     +2.74pp
Sonnet 4.6              0.04%    +14.50pp
Opus 4.7                0.00%     +0.20pp
Gemini 2.5 Flash        0.04%    +14.40pp
Gemini 2.5 Pro          0.00%    +63.40pp
Gemini 3 Flash          

## LaTeX summary table

In [ ]:
def fmt_p(p):
    if p == 0.0:
        return '$< 10^{-16}$'
    if p < 1e-4:
        exp = int(np.floor(np.log10(p)))
        man = p / (10 ** exp)
        return f'${man:.1f}{{\\times}}10^{{{exp}}}$'
    return f'${p:.4f}$'

def sig_star(p):
    if p < 0.001: return '***'
    if p < 0.01: return '**'
    if p < 0.05: return '*'
    return ''

lines = []
# [!ht] keeps this table inline-ish in the appendix, preventing it from
# floating up above the O*NET Task Inventory promptboxes (which are
# inline / non-floating) and breaking the intended section order.
lines.append('\\begin{table}[!ht]')
lines.append('\\centering')
lines.append('\\small')
lines.append('\\begin{tabular}{llrrrrl}')
lines.append('\\toprule')
lines.append('\\textbf{Test} & \\textbf{Comparison} & \\textbf{$\\boldsymbol{n}$} & \\textbf{Statistic} & \\textbf{$\\boldsymbol{p}_{\\textbf{raw}}$} & \\textbf{$\\boldsymbol{p}_{\\textbf{Holm}}$} & \\textbf{Sig.} \\\\')
lines.append('\\midrule')

# Test A — handle all-identical ρ case with exclusion count
if np.isnan(W_a):
    lines.append(f'A: Trend & Tiers 1--3 $\\rho$ & {len(rho_arr)} '
                 f'& $\\rho = {rho_arr[0]:.1f}$ (all) & --- & --- & \\checkmark \\\\')
else:
    med_rho = float(np.median(rho_arr))
    lines.append(f'A: Trend & Tiers 1--3 $\\rho$ & {len(rho_arr)} '
                 f'& $W = {W_a}$, median $\\rho = {med_rho:+.2f}$ '
                 f'& {fmt_p(p_a)} & --- & {sig_star(p_a) or "NS"} \\\\')

# Test B
lines.append('\\midrule')
for r in test_b_results:
    lines.append(f'B: Lift & T{r["tier"]} vs T3 & {r["n"]} '
                 f'& $W = {r["W"]:.0f}$ & {fmt_p(r["p_raw"])} & {fmt_p(r["p_corr"])} & {r["sig"]} \\\\')

# Test C — ALL pairs including NS
lines.append('\\midrule')
for r in test_c_results:
    sig_label = r['sig'] if r['sig'] != 'NS' else 'NS'
    lines.append(f'C: Pair & T{r["ti"]} vs T{r["tj"]} & {N} '
                 f'& $W = {r["W"]:.0f}$ & {fmt_p(r["p_raw"])} & {fmt_p(r["p_corr"])} & {sig_label} \\\\')

# Pressure susceptibility — continuous Spearman only
lines.append('\\midrule')
lines.append(f'Suscept. & Spearman $\\rho$ & {N} '
             f'& $\\rho = {rho_cap:.2f}$ & {fmt_p(p_cap)} & --- & {sig_star(p_cap) or "NS"} \\\\')

lines.append('\\bottomrule')
lines.append('\\end{tabular}')

# Caption: describe Test A honestly (may be non-significant trend if median < 1.0)
caption_a_note = (f'{n_nan} models excluded from Test~A due to constant zero ISR. '
                  if n_nan > 0 else '')
caption_a_desc = (f'Test~A: per-model Spearman $\\rho$ across tiers 1--3 '
                  f'($n = {len(rho_arr)}$; {caption_a_note}')
if np.isnan(W_a):
    caption_a_desc += f'all show $\\rho = {rho_arr[0]:.1f}$). '
else:
    med_rho = float(np.median(rho_arr))
    caption_a_desc += f'median $\\rho = {med_rho:+.2f}$, tested against '
    caption_a_desc += '$H_0$: median $\\rho = 0$). '

lines.append(f'\\caption{{Escalation ladder statistical tests (no-safety, $T{{=}}0.0$). '
             f'{caption_a_desc}'
             f'Test~B: Wilcoxon signed-rank on lift '
             f'$\\Delta_k = \\text{{ISR}}(k) - \\text{{ISR}}(3)$, $n = {N}$. '
             f'Test~C: pairwise Wilcoxon between pressure tiers, $n = {N}$. '
             f'Tests B and C use Holm--Bonferroni correction. '
             f'Suscept.: Spearman $\\rho$ between baseline vulnerability '
             f'(Tier~3 ISR) and max pressure lift.}}')
lines.append('\\label{tab:escalation-stats}')
lines.append('\\end{table}')

tex = '\n'.join(lines) + '\n'
out = TABLES_DIR / 'escalation_stats.tex'
out.write_text(tex)
print(f'Wrote {out}')
print()
print(tex)